In [0]:
dim = ['patients', 'providers','payers', 'organizations']
fact = ['encounters', 'medications', 'immunizations','procedures']
# ref = ['conditions','allergies','observations','supplies','careplans','devices','imaging_studies', 'payer_transition'] unnecessary we dont need these here

In [0]:
# building dim tables
from pyspark.sql.functions import *
patients_df = spark.read.table("syntheaproject.silver_manual.patients")

dim_patient_df = (patients_df.withColumn("patient_sk",monotonically_increasing_id()).withColumn("full_name",concat_ws(" ",col("first_name"),col("last_name"))).withColumn("age",floor(months_between(current_date(),col("birth_date")) / 12)))

dim_patient_df = (dim_patient_df.select("patient_sk","pat_id","full_name","birth_date","death_date","age","gender","race","ethnicity","marital","city","state","zip_code","healthcare_expenses","healthcare_coverage"))

(dim_patient_df.write.format("delta").mode("overwrite").saveAsTable("syntheaproject.gold_manual.dim_patient"))

In [0]:
providers_df = spark.read.table("syntheaproject.silver_manual.providers")

dim_provider_df = (providers_df.withColumn("provider_sk",monotonically_increasing_id()))

dim_provider_df = (dim_provider_df.select("provider_sk","prov_id", "name", "org_id", "gender", "speciality", "utilization"))

(dim_provider_df.write.format("delta").mode("overwrite").saveAsTable("syntheaproject.gold_manual.dim_provider"))

In [0]:
payers_df = spark.read.table("syntheaproject.silver_manual.payers")

dim_payer_df = (payers_df.withColumn("payer_sk",monotonically_increasing_id()))

dim_payer_df = (dim_payer_df.select("payer_sk","pay_id", "name", "amount_covered", "amount_uncovered", "revenue", "covered_encounters", "uncovered_encounters", "covered_medications", "uncovered_medications", "covered_procedures", "uncovered_procedures", "covered_immunizations", "uncovered_immunizations", "unique_customers"))

(dim_payer_df.write.format("delta").mode("overwrite").saveAsTable("syntheaproject.gold_manual.dim_payer"))

In [0]:
organizations_df = spark.read.table("syntheaproject.silver_manual.organizations")

dim_organization_df = (organizations_df.withColumn("organization_sk",monotonically_increasing_id()))

dim_organization_df = (dim_organization_df.select("organization_sk","org_id", "name", "city", "state", "revenue", "utilization"))

(dim_organization_df.write.format("delta").mode("overwrite").saveAsTable("syntheaproject.gold_manual.dim_organization"))

In [0]:
# Event fact table build
encounters_df = spark.read.table("syntheaproject.silver_manual.encounters")
patients_age_df = (patients_df.select("pat_id","birth_date"))

fact_encounter_df = (encounters_df.alias("e").join(patients_age_df.alias("p"),on="pat_id",how="left"))

fact_encounter_df = (fact_encounter_df.withColumn("age_at_encounter",floor(months_between(col("start"),col("birth_date")) / 12)))

fact_encounter_df = (fact_encounter_df.select(col("e.enc_id").alias("enc_id"),col("e.pat_id").alias("pat_id"),col("e.org_id").alias("org_id"),col("e.prov_id").alias("prov_id"),col("e.pay_id").alias("pay_id"),col("e.start").alias("start"),col("e.stop").alias('stop'),col("e.encounterclass").alias('class'),col("e.total_claim_cost").alias('cost'),col("e.payer_coverage").alias('covered'),col("age_at_encounter").alias('age')))

(fact_encounter_df.write.format("delta").mode("overwrite").saveAsTable("syntheaproject.gold_manual.fact_encounter"))

In [0]:
immunization_df = spark.read.table("syntheaproject.silver_manual.immunizations")
patients_age_df = (patients_df.select("pat_id","birth_date"))

fact_immunization_df = (immunization_df.alias("i").join(patients_age_df.alias("p"),on="pat_id",how="left"))

fact_immunization_df = (fact_immunization_df.withColumn("age_at_immunization",floor(months_between(col("date"),col("birth_date")) / 12)))

fact_immunization_df = (fact_immunization_df.select(col("i.pat_id").alias('pat_id'), col("i.enc_id").alias("enc_id"), col("i.code").alias("code"), col("i.date").alias('date'), col("i.base_cost").alias("cost"), col("age_at_immunization").alias('age')))

(fact_immunization_df.write.format("delta").mode("overwrite").saveAsTable("syntheaproject.gold_manual.fact_immunization"))

In [0]:
medications_df = spark.read.table("syntheaproject.silver_manual.medications")
fact_medication_df = (medications_df.select("pat_id", "payer", "enc_id", "start", "stop", "payer_coverage", "totalcost"))
fact_medication_df.write.format("delta").mode("overwrite").saveAsTable("syntheaproject.gold_manual.fact_medication")

In [0]:
procedures_df = spark.read.table("syntheaproject.silver_manual.procedures")
fact_procedure_df = (procedures_df.select("pat_id", "enc_id", "code", "date", "base_cost"))
fact_procedure_df.write.mode("overwrite").saveAsTable("syntheaproject.gold_manual.fact_procedure")